# Отчёт симуляции: числа стадий и их происхождение

Для ветки `results/<BCR_DATASET>/simulated/<BCR_BRANCH>` собирает таблицу стадий (эталон → ПЦР1 → порция → фрагментация → ПЦР2 → риды) с формулой для каждого числа и пишет `qc/simulation_report.md`.


In [ ]:
"""Отчёт по ветке симуляции: числа каждой стадии и откуда они берутся (формулы с фактическими значениями).

Запуск: BCR_DATASET=PRJEB30386|ERP003950 BCR_BRANCH=<ветка> jupyter execute simulation_report.ipynb  → пишет <branch>/qc/simulation_report.md
"""
import csv, json, sys, statistics
from pathlib import Path
import numpy as np

import os
DS_DIR = Path(os.environ.get("BCR_VOLUME", "/data/user/epishkin")) / "results" / os.environ["BCR_DATASET"]
BRANCH = os.environ["BCR_BRANCH"]
B = DS_DIR / "simulated" / BRANCH
HUMAN = DS_DIR.name == "PRJEB30386"


def tsv(p):
    with open(p, newline="") as h:
        return list(csv.DictReader(h, delimiter="\t"))


def f(x, d=0):
    return f"{x:,.{d}f}".replace(",", " ")


def pct(a, b, d=1):
    return f"{100 * a / b:.{d}f}%"


def sci(x):
    m, e = f"{x:.2e}".split("e")
    return f"{m}·10^{int(e)}"


def pcr_gain(cycles, mean=0.85, conc=80.0, n=200_000):
    e = np.random.default_rng(0).beta(mean * conc, (1 - mean) * conc, n)
    return float(np.mean((1 + e) ** cycles))


samples = sorted({p.name.split("_template_qc")[0] for p in (B / "00_primary_truth").glob("*_template_qc.tsv")})
fs = json.loads((DS_DIR / "post_annotation_filtered" / "filter_summary.json").read_text())
out = [f"# Отчёт симуляции `{BRANCH}`\n",
       "Числа посчитаны из файлов ветки; в колонке «Откуда» — формула с фактическими значениями.\n"]

for s in samples:
    tq = tsv(B / "00_primary_truth" / f"{s}_template_qc.tsv")
    pcr1 = tsv(B / "01_pcr1" / f"{s}_pcr1_pool.tsv")
    lib = tsv(B / "02_fragmentation" / f"{s}_library_summary.tsv")[0]
    model = tsv(B / "02_fragmentation" / f"{s}_fragment_length_model.tsv")
    alloc = tsv(B / "04_read_allocation" / f"{s}_allocation.tsv")
    recon = {r["locus"]: r for r in tsv(B / "qc" / f"{s}_reconstructability_summary.tsv")}
    final = B / "qc" / "final_qc.tsv"

    runs = ["ERR3004229", "ERR3004230", "ERR3004231", "ERR3004232"] if HUMAN else [s]
    inp = sum(fs["samples"][r]["input"] for r in runs)
    passed = sum(fs["samples"][r]["passed"] for r in runs)
    if HUMAN:
        audit = tsv(B / "qc" / "source_truth_audit_by_run.tsv")
        eligible = sum(int(a["simulation_eligible_rows"]) for a in audit)
        umi = {r["umi_class"]: int(r["umis"]) for r in tsv(B / "qc" / "umi_consensus_audit.tsv")}
        n0 = sum(int(r["unique_umis"]) for r in tq)
    else:
        audit = tsv(B / "qc" / f"{s}_source_truth_audit.tsv")[0]
        eligible = int(audit["simulation_eligible_rows"])
        n0 = sum(int(r["observed_multiplicity"]) for r in tq)
    ver = [r for r in tq if r["verified"] == "True"]
    copies_col = "unique_umis" if HUMAN else "observed_multiplicity"
    n0_ver = sum(int(r[copies_col]) for r in ver)
    tlen = statistics.median(int(r["length"]) for r in tq)
    vjlen = statistics.median(int(r["vj_end"]) - int(r["vj_start"]) for r in tq)

    p1 = sum(int(r["pcr_copies"]) for r in pcr1)
    g1_theory = pcr_gain(25)
    a = int(lib["library_input_molecules"])
    mean_len = float(lib["pcr1_weighted_mean_template_length"])
    q = float(lib["expected_retained_fragments_per_input_molecule"])
    g2 = pcr_gain(10)
    p2 = a * q * g2
    r_pairs = int(lib["read_pairs"])
    raw = r_pairs / 3
    tw = int(lib["templates_with_reads"])
    ins = np.repeat([int(x["fragment_length"]) for x in alloc], [int(x["simulated_read_pairs"]) for x in alloc])
    ver_ids = {r["template_id"] for r in ver}
    ver_read = len({x["template_id"] for x in alloc} & ver_ids)
    loc_pairs = {}
    for x in alloc:
        loc_pairs[x["locus"]] = loc_pairs.get(x["locus"], 0) + int(x["simulated_read_pairs"])
    q_len = [float(m["expected_retained_per_molecule"]) for m in model]

    rows = [
        ("Склеенные риды на входе фильтра", f(inp), "—",
         f"pRESTO AssemblePairs, сумма по {'4 библиотекам' if HUMAN else 'образцу'} (filter_summary.json)"),
        ("Прошли фильтр аннотации", f"{f(passed)} ({pct(passed, inp)})", "—",
         "продуктивные, полная V…J, V с 1-й позиции, ожидаемый локус, пороги identity/support"),
        ("Годны для симуляции", f(eligible), "—",
         f"{f(passed)} − {f(passed - eligible)} " + ("(нет валидного 20-нт UMI или есть N)" if HUMAN else "(есть N)")),
    ]
    if HUMAN:
        rows.append(("Исходные молекулы = UMI", f(n0), f"{f(len(tq))} шаблонов",
                     f"каждый UMI = 1 молекула. Консенсус ≥2 ридов: {f(umi.get('consensus_verified', 0))}; "
                     f"1 рид или без консенсуса: {f(umi.get('single_read', 0) + umi.get('no_consensus', 0))}, из них "
                     f"{f(umi.get('unverified_corrected_to_consensus_1nt', 0))} в 1 замене от проверенного консенсуса → копия этого консенсуса"))
    else:
        rows.append(("Исходные молекулы = риды после схлопывания", f(n0), f"{f(len(tq))} шаблонов",
                     f"UMI нет: {f(int(audit['unique_exact_vj']))} точных вариантов V…J, {f(int(audit['absorbed_error_variants']))} "
                     f"поглощены соседом в 1 замене, встречающимся ≥3× чаще; копии шаблона = его риды"))
    rows += [
        ("Проверенные шаблоны (эталон оценки)", f"{f(n0_ver)} молекул ({pct(n0_ver, n0)})", f"{f(len(ver))} ({pct(len(ver), len(tq))})",
         "консенсус UMI подтверждён" if HUMAN else "≥2 рида после схлопывания; остальные симулируются, но не оцениваются"),
        ("Длина шаблона", "—", "—", f"медиана {f(tlen)} нт (ампликон с праймерами), из них V…J {f(vjlen)} нт"),
        ("После ПЦР1 (25 циклов)", sci(p1), f(len(pcr1)),
         f"каждый цикл копия с вероятностью e ~ Beta(ср. 0.85): P1/N0 = {sci(p1 / n0)} на молекулу "
         f"(E[(1+e)^25] = {sci(g1_theory)}; больше 1.85^25 = {sci(1.85 ** 25)} из-за разброса e)"),
        (f"Порция на фрагментацию ({lib['library_input_ng']} нг)", f(a), f"{f(int(lib['library_input_templates']))} шаблонов",
         f"A = {lib['library_input_ng']} нг / ({f(mean_len, 1)} нт × 650 г/моль) × 6.022·10^23 = {sci(a)}; "
         f"на исходную молекулу A/N0 = {f(a / n0)} копий; доля пула A/P1 = {pct(a, p1, 4)}"),
        ("Фрагментация (1 разрез) + отбор 250±40 нт", f"≈ {sci(a * q)} кусков", "—",
         f"q = {q:.4f} отобранных кусков на молекулу = среднее по точкам разреза p(левый)+p(правый), "
         f"p(L) = exp(−½((L−250)/40)²), L < 152 → 0; q по длинам шаблона {min(q_len):.3f}–{max(q_len):.3f}"),
        ("После ПЦР2 (10 циклов)", f"≈ {sci(p2)}", "—", f"A·q × E[(1+e)^10] = {sci(a * q)} × {g2:.0f}"),
        ("Прочитано (пары)", f"{f(r_pairs)} ({pct(r_pairs, p2, 4)} пула ПЦР2)", f"{f(int(lib['fragment_species_with_reads']))} видов кусков",
         f"R = 3 × сырые пары исходных данных = 3 × {f(raw)}; на исходную молекулу R/N0 = {r_pairs / n0:.1f} пар; "
         f"пар на вид куска {lib['read_pairs_per_fragment_species']}"),
        ("Шаблонов с ≥1 парой", "—", f"{f(tw)} ({pct(tw, len(tq))}); проверенных {f(ver_read)} ({pct(ver_read, len(ver))})",
         "риды делятся между шаблонами ∝ копиям в порции × q; редкие шаблоны могут не получить ни одной пары"),
        ("Пары по локусам", " · ".join(f"{k} {f(v)} ({pct(v, r_pairs)})" for k, v in sorted(loc_pairs.items())), "—",
         "доля локуса = доля его исходных библиотек в сырых парах" if HUMAN else "только IGH"),
        ("Длина вставки", f"медиана {f(np.median(ins))} нт (5–95%: {f(np.percentile(ins, 5))}–{f(np.percentile(ins, 95))})", "—",
         f"{pct((ins > 300).sum(), len(ins))} пар со вставкой >300 нт — непрочитанный промежуток между R1 и R2"),
    ]
    fq = [x for x in tsv(final) if x["sample"] == s] if final.exists() else []
    rows.append(("Итоговые FASTQ", f"{f(int(fq[0]['R1_reads']))} × R1/R2, {fq[0]['R1_lengths']} нт" if fq else "генерация ещё идёт",
                 "—", f"final_qc valid = {fq[0]['valid']}" if fq else ""))

    out.append(f"\n## {s}\n\n| Этап | Молекул | Уникальных | Откуда |\n|---|---|---|---|")
    out += [f"| {a_} | {b_} | {c_} | {d_} |" for a_, b_, c_, d_ in rows]
    cls = ["full_coverage_read_overlap", "full_coverage_mate_linked_only", "full_coverage_unlinked",
           "partial_read_coverage", "no_reads"]
    names = ["связаны перекрытием ридов", "только mate-парой", "не связаны", "часть V…J не прочитана", "нет ридов"]
    r_all = recon["ALL"]
    out.append("\n**Восстановимость проверенных шаблонов** (что физически есть в ридах):\n\n| класс | шаблонов | доля |\n|---|---:|---:|")
    out += [f"| {n} | {f(int(r_all[c]))} | {float(r_all[c + '_frac']):.1%} |" for c, n in zip(cls, names)]

out.append("""
## Откуда параметры

| параметр | значение | источник |
|---|---|---|
| исходные копии | UMI (человек) / риды после схлопывания ошибок (мышь) | данные |
| циклы ПЦР1 / ПЦР2 | 25 / 10 | протокол 5′MTPX (у мыши Greiff 2014 — 28 циклов одной ПЦР, модель общая); на итог не влияет: порции берутся пропорционально составу пула |
| эффективность цикла | Beta, среднее 0.85, концентрация 80 | допущение |
| порция на фрагментацию | 5 нг продукта ПЦР1 | протокол: 5–10 нг |
| фрагментация | 1 равновероятный разрез на молекулу | выбор ветки random_cut (многократные разрывы — ветка ultrasonic) |
| отбор по длине | 250±40 нт, мягкий Гаусс, < 152 нт отбрасывается | выбор (250 — центр); 152 — InSilicoSeq генерирует риды 151 нт и требует кусок длиннее |
| глубина | 3 × сырые пары исходных данных | выбор |
| риды | InSilicoSeq, профиль NovaSeq, PE151 → обрезка до PE150 | стандартный профиль платформы |

Абсолютные размеры пулов (P1, P2) и доли «порция/пул», «прочитано/пул» — производные числа: в модели нет насыщения ПЦР, поэтому пулы завышены, но состав пула и итог от них не зависят. Результат определяют копии на исходную молекулу в порции (A/N0) и пары ридов на молекулу (R/N0).
""")
text = "\n".join(out)
(B / "qc" / "simulation_report.md").write_text(text)
print(text)
